<a href="https://colab.research.google.com/github/mitalidaduria/nlp-payments-lab/blob/main/MLflow_Experiment_Tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install mlflow xgboost optuna scikit-learn pandas joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [6]:
import os
import joblib
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, auc, f1_score

# ------------------------------------------------------------------
# 1. Dataset Generation & Setup
# ------------------------------------------------------------------
N_SAMPLES = 5000
FRAUD_RATE = 0.05
N_FEATURES = 15

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=10,
    n_redundant=2,
    weights=[1 - FRAUD_RATE, FRAUD_RATE],
    random_state=42
)

feature_names = [f"feature_{i}" for i in range(N_FEATURES)]
X_df = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42, stratify=y
)

# Build & fit feature scaling pipeline
scaler_pipeline = Pipeline([
    ('scaler', StandardScaler())
])
X_train_scaled = scaler_pipeline.fit_transform(X_train)
X_test_scaled = scaler_pipeline.transform(X_test)

# ------------------------------------------------------------------
# 2. Set MLflow Experiment
# ------------------------------------------------------------------
mlflow.set_experiment("Payment_Fraud_Detection")

# ------------------------------------------------------------------
# 3. MLflow Run Execution
# ------------------------------------------------------------------
with mlflow.start_run(run_name="xgb_optuna_50trials"):

    # A. Log Data Configuration Parameters
    mlflow.log_param("n_samples", N_SAMPLES)
    mlflow.log_param("fraud_rate", FRAUD_RATE)
    mlflow.log_param("n_features", N_FEATURES)

    # B. Define Optuna Objective Function
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 9),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'eval_metric': 'logloss',
            'random_state': 42
        }

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_train_scaled, y_train)

        preds_probs = clf.predict_proba(X_test_scaled)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, preds_probs)
        return auc(recall, precision)

    # Disable Optuna verbosity for clean terminal output
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50)

    # C. Log Optuna Best Hyperparameters
    best_params = study.best_params
    for param_name, param_val in best_params.items():
        mlflow.log_param(f"best_{param_name}", param_val)

    # D. Train Final Model with Best Parameters
    final_model = xgb.XGBClassifier(**best_params, random_state=42)
    final_model.fit(X_train_scaled, y_train)

    # E. Evaluate Final Model & Log Metrics
    y_probs = final_model.predict_proba(X_test_scaled)[:, 1]
    y_preds = final_model.predict(X_test_scaled)

    precision, recall, _ = precision_recall_curve(y_test, y_probs)
    pr_auc_score = auc(recall, precision)
    f1 = f1_score(y_test, y_preds)

    mlflow.log_metric("pr_auc", pr_auc_score)
    mlflow.log_metric("f1_score", f1)

    # F. Save & Log Feature Pipeline Artifact
    os.makedirs("temp_artifacts", exist_ok=True)
    pipeline_path = "temp_artifacts/feature_pipeline.pkl"
    joblib.dump(scaler_pipeline, pipeline_path)
    mlflow.log_artifact(pipeline_path, artifact_path="preprocessing")

    # G. Log XGBoost Model Artifact
    mlflow.xgboost.log_model(final_model, name="model")

    print(f"Run complete! PR-AUC: {pr_auc_score:.4f} | F1 Score: {f1:.4f}")

Run complete! PR-AUC: 0.8247 | F1 Score: 0.6897


In [7]:
import subprocess
from google.colab import output

# Start MLflow UI server in the background
subprocess.Popen(["mlflow", "ui", "--port", "5000"])

# Render the dashboard directly inside Colab
output.serve_kernel_port_as_iframe(5000)

<IPython.core.display.Javascript object>

In [12]:
import subprocess
from google.colab import output

# Start MLflow UI server listening on 0.0.0.0 to accept Colab's proxy domain
subprocess.Popen(["mlflow", "ui", "--host", "0.0.0.0", "--port", "5000"])

# Render the interactive UI inside Colab
output.serve_kernel_port_as_iframe(5000)

<IPython.core.display.Javascript object>

In [14]:
import subprocess
from google.colab import output

# Terminate any existing MLflow servers
!pkill -f "mlflow ui"

# Start MLflow listening on 0.0.0.0 to accept requests through Colab
subprocess.Popen(["mlflow", "ui", "--host", "0.0.0.0", "--port", "5000"])

# Display the interactive MLflow UI inside Colab
output.serve_kernel_port_as_iframe(5000)

<IPython.core.display.Javascript object>

In [18]:
import subprocess
import os
from google.colab import output

# 1. Kill any stubborn background servers
!pkill -f "mlflow"

# 2. Force MLflow to accept ALL incoming host headers (bypasses the DNS error entirely)
os.environ["MLFLOW_SERVER_ALLOWED_HOSTS"] = "*"
os.environ["MLFLOW_SERVER_CORS_ALLOWED_ORIGINS"] = "*"

# 3. Start the tracking server
subprocess.Popen(["mlflow", "server", "--host", "0.0.0.0", "--port", "5000"])

# 4. Open the dashboard in a new tab!
output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>